In [105]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from pandas_ta.volatility import kc 
import enum

class EMA25(enum.Enum):
    PRICE_ACCION_NEUTRAL = 0
    PRICE_ACCION_UNDER = 1
    PRICE_ACCION_OVER = 2

def calculateBolingerAndKeltnerChannels(kc):
    # Bolinger Bands
    df.ta.bbands(append=True, length=20, std=2)
    
    # Initialize Keltner Channel Indictor
    kc=kc(high=df['High'], low=df['Low'], close=df["Close"], window=20)
    
    #Bolinger Band Upper - Keltner Channel Upper
    df['bbu_minus_kcu'] = df['BBU_20_2.0'] - kc['KCUe_20_2']
    
def detectPosibleBreak(window = 3):

    df['posibleBreak'] = EMA25['PRICE_ACCION_NEUTRAL'].value

    '''
    OVER PRICE ACCION
    '''
    cond = (df.squeezeCloseToEma == EMA25['PRICE_ACCION_UNDER'].value)
    cond2 = (len(cond.tail(3).values) == window)
    cond3 = (cond2 & (df.Close<df.ema25))
    cond4 = (cond3 & (df.squeezeCloseToEma))
    df.loc[cond4, 'posibleBreak'] = EMA25['PRICE_ACCION_UNDER'].value

    '''
    UNDER PRICE ACCION
    '''
    cond = (df.squeezeCloseToEma == EMA25['PRICE_ACCION_OVER'].value)
    cond2 = (len(cond.tail(3).values) == window)
    cond3 = (cond2 & (df.Close>df.ema25))
    cond4 = (cond3 & (df.squeezeCloseToEma))
    df.loc[cond4, 'posibleBreak'] = EMA25['PRICE_ACCION_OVER'].value

def detectSqueezeCloseToEMA(zoneWidth = .30):

    df['squeezeCloseToEma'] = EMA25['PRICE_ACCION_NEUTRAL'].value
    
    cond =((df.isSqueezed == True) & (abs(df.Low-df.ema25)<=zoneWidth))
    df.loc[cond, 'isqueezeCloseToEma'] = EMA25['PRICE_ACCION_OVER'].value

    cond =((df.isSqueezed == True) & (abs(df.High-df.ema25)<=zoneWidth))
    df.loc[cond, 'squeezeCloseToEma'] = EMA25['PRICE_ACCION_UNDER'].value
    
def detectSqueeze():
    
    df['isSqueezed'] = EMA25['PRICE_ACCION_NEUTRAL'].value
    cond = ((df.bbu_minus_kcu <= 0) & (df.Close < df.ema25))
    df.loc[cond, 'isSqueezed'] = EMA25['PRICE_ACCION_UNDER'].value
    
    cond2 =((df.bbu_minus_kcu <= 0) & (df.Close > df.ema25))
    df.loc[cond2, 'isSqueezed'] = EMA25['PRICE_ACCION_OVER'].value

if __name__ == '__main__':
    
    df = pd.read_csv("../data/MSFT/MSFT.USUSD_Candlestick_5_M_BID_01.09.2022-21.09.2024.csv")
    df=df[0:10000]
    df.rename(columns = {'Gmt time':'datetime'}, inplace = True)
    df["datetime"]=df["datetime"].str.replace(".000","")
    df['datetime']=pd.to_datetime(df['datetime'],format='%d.%m.%Y %H:%M:%S')    
    df['ema25'] = df['Close'].ewm(span=25, adjust=False).mean()
    df.reset_index(drop=True, inplace=True)
    pd.options.mode.copy_on_write = True

    calculateBolingerAndKeltnerChannels(kc)
    detectSqueeze()
    detectSqueezeCloseToEMA()
    detectPosibleBreak()
    
    df = df[["datetime","Open","High","Low","Close","isSqueezed","squeezeCloseToEma",'posibleBreak']]
    result = df.loc[df["posibleBreak"]==0]
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5554 entries, 0 to 9847
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   datetime           5554 non-null   datetime64[ns]
 1   Open               5554 non-null   float64       
 2   High               5554 non-null   float64       
 3   Low                5554 non-null   float64       
 4   Close              5554 non-null   float64       
 5   isSqueezed         5554 non-null   int64         
 6   squeezeCloseToEma  5554 non-null   int64         
 7   posibleBreak       5554 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(3)
memory usage: 390.5 KB
